# 8.2 Institutional Pattern Discovery II: Unsupervised Learning with Survey Data — Code Brief

## Key Concepts

- K-Means clustering on the Module 7 master matrix (academic + demographic + survey/text PCA features) to discover natural student groupings.
- Elbow method (inertia vs. K) picks the number of clusters.
- Clusters are profiled using raw (unscaled) values for interpretability, and validated with sample free-text comments per cluster ("student voice").

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(15)
random.seed(15)

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
df_ml_train = pd.read_csv(f'{filepath}ML_SURVEY_MASTER_TRAIN.csv')

df_ml_train

In [ ]:
X_train = df_ml_train.drop(columns = ['SEM_3_STATUS'])

In [ ]:
df_training = pd.read_csv(f'{filepath}training.csv')
df_training1 = df_training.drop(columns = ['SEM_3_STATUS'])
X_raw = pd.concat([df_training1,X_train.iloc[:,19:]],axis=1)
X_raw

## Choosing K (Elbow Method)

In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertia.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertia,
              markers=True,
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'},
              title='Elbow Method — Choosing K for K-Means')
fig.show()
print("Look for the point where the curve flattens — that is your elbow.")


## Fit K-Means & Assign Cluster Labels

In [ ]:
OPTIMAL_K = 3  # ← update this based on the elbow plot above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans.fit(X_train)

X_train = X_train.copy()
X_raw['Cluster'] = kmeans.labels_

print("Cluster sizes:")
print(X_raw['Cluster'].value_counts().sort_index())


## Visualize Clusters with PCA

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(X_train)

df_plot = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': X_raw['Cluster'].astype(str),
    'GPA': X_raw['HS_GPA'].round(2),
    'First_Gen': X_raw['FIRST_GEN_STATUS'],
    'Gender': X_raw['GENDER']
}, index=X_raw.index)

fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='Cluster',
    hover_data=['GPA', 'First_Gen', 'Gender'],
    title=f'K-Means Clusters (K={OPTIMAL_K}) — Visualized with 2D PCA',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'}
)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.show()


## Cluster Profiling

In [ ]:
# Numeric profile
profile_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
                'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
profile_cols = [c for c in profile_cols if c in X_raw.columns]

cluster_profile = X_raw.groupby('Cluster')[profile_cols].mean().round(3)
print("── Cluster Numeric Profile ──")
cluster_profile


In [ ]:
# Categorical breakdown
print("── First-Generation Status by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['FIRST_GEN_STATUS'], normalize='index').round(3)


In [ ]:
print("\n── Gender by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['GENDER'], normalize='index').round(3)

In [ ]:
# Sample comments from each cluster

ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')

print("── Representative Comments by Cluster ──")
for c in sorted(X_raw['Cluster'].unique()):
    index = X_raw[X_raw['Cluster'] == c].index
    subset = ML_Survey_Data.iloc[index,:]
    examples = subset.sample(min(2, len(subset)), random_state=42)['Free_Response_Text'].tolist()
    print(f"\nCluster {c} (n={len(subset)}):")
    for ex in examples:
        print(f"  • {ex}")

In [ ]:
# Visualize numeric profile as a heatmap
import plotly.figure_factory as ff

z = cluster_profile.values.tolist()
x = cluster_profile.columns.tolist()
y = [f'Cluster {i}' for i in cluster_profile.index]

fig = px.imshow(
    cluster_profile,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Blues',
    title='Cluster Profiles — Average Feature Values',
    labels={'x': 'Feature', 'y': 'Cluster', 'color': 'Mean Value'}
)
fig.show()


## Save the Cluster Model

In [ ]:
import pickle
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'

# Define the filename for the pickled model
kmeans_cluster_model = f'{model_filepath}kmeans_model.pkl'

# Pickle out the kmeans model
with open(kmeans_cluster_model, 'wb') as file:
    pickle.dump(kmeans, file)

print(f"KMeans model successfully pickled to {kmeans_cluster_model}")